# 03. 채널 구조 탐색: 연관 · 대체 · 외환의 독립성
> 계획서: `계획/P3_채널구조탐색.md`

**질문**
1. 건수 변수들은 어떤 구조(차원)를 가지나? (온라인 vs 오프라인, 자동이체, 외환)
2. 채널 간 **대체(이동)** 가 있나? (한 채널이 줄 때 다른 채널이 느는가)
3. 외환은 다른 채널과 독립된 축인가?

**표본**: P36 과 P35+ 를 같이 돌려 결과가 같은지 비교합니다 (빈 달은 M0: 관측된 달만 사용).

**주의**: 기업이 크면 모든 채널을 많이 쓰므로 *기업 간* 상관은 크기 효과입니다. 대체 관계는 **기업 안에서 시간에 따른 변화**(기업 평균·월 평균을 뺀 값)로 봅니다.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "건수분석" / "code", base / "yangdaecheon" / "건수분석" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 yangdaecheon/건수분석/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
pops = {"P36": masks["P36"], "P35+": masks["P35p"]}
print({k: int(v.sum()) for k, v in pops.items()}, "개 기업")

## 1. 기업 간 상관 (기업 평균 코드의 Spearman) — 크기 효과가 섞여 있음

In [ ]:
def heat(M, title, vmin=-1, vmax=1):
    fig, ax = plt.subplots(figsize=(6.4, 5.2))
    im = ax.imshow(M.values, vmin=vmin, vmax=vmax, cmap="RdBu_r")
    ax.set_xticks(range(len(M))); ax.set_xticklabels(M.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index)
    for i in range(len(M)):
        for j in range(len(M)):
            ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()

def firm_means(mask):
    return pd.DataFrame({cl.SHORT[c]: np.nanmean(cl.codes_float(panel.cnt[c][mask], panel.observed[mask]), axis=1) for c in cl.CNT_COLS})

B36 = firm_means(pops["P36"]).corr(method="spearman")
cl.show_table(B36.round(2).reset_index().rename(columns={"index": ""}), "기업 간 Spearman 상관 (P36)", OUT / "nb03", "between_corr_P36")
heat(B36, "기업 간 상관 (P36): 크기 효과 포함")

## 2. 기업 안에서의 상관 (기업·월 평균을 뺀 변화) — 대체 관계는 여기서 봅니다

In [ ]:
def within(mask):
    cols = {cl.SHORT[c]: cl.two_way_demean(cl.codes_float(panel.cnt[c][mask], panel.observed[mask])).ravel() for c in cl.CNT_COLS}
    D = pd.DataFrame(cols)
    return D.corr(), D

W36, D36 = within(pops["P36"])
W35, D35 = within(pops["P35+"])
cl.show_table(W36.round(2).reset_index().rename(columns={"index": ""}), "기업 내 상관 (P36)", OUT / "nb03", "within_corr_P36")
heat(W36, "기업 내 상관 (P36): 음수면 대체 후보")
off = ~np.eye(len(W36), dtype=bool)
maxdiff = float(np.nanmax(np.abs((W35 - W36).values[off])))
print(f"P35+ 와 P36 의 기업 내 상관 최대 차이(비대각): {maxdiff:.3f}")

## 3. 차원 구조 (기업 내 상관행렬의 주성분)

In [ ]:
C = W36.fillna(0).values
vals, vecs = np.linalg.eigh(C)
o = np.argsort(vals)[::-1]; vals, vecs = vals[o], vecs[:, o]
ev = pd.DataFrame({"성분": [f"PC{i + 1}" for i in range(len(vals))], "고유값": vals.round(3),
                   "설명 비율(%)": (vals / vals.sum() * 100).round(1), "누적(%)": (np.cumsum(vals) / vals.sum() * 100).round(1)})
cl.show_table(ev, "주성분 설명력 (기업 내 변화, P36)", OUT / "nb03", "pca_variance")
load = pd.DataFrame(vecs[:, :3] * np.sqrt(np.maximum(vals[:3], 0)), index=W36.index, columns=["PC1", "PC2", "PC3"]).round(2)
cl.show_table(load.reset_index().rename(columns={"index": "변수"}), "주성분 적재값 (P36)", OUT / "nb03", "pca_loadings")

## 4. 채널 이용 추이: 사용 비율이 시간에 따라 어떻게 바뀌나 (P36)

In [ ]:
P = pops["P36"]
use = {cl.SHORT[c]: np.nanmean(np.where(panel.cnt[c][P] >= 0, panel.cnt[c][P] > 0, np.nan), axis=0) * 100 for c in cl.CNT_COLS[:6]}
fig, ax = plt.subplots(figsize=(9, 3.8))
for k, v in use.items():
    ax.plot(range(cl.T), v, lw=2, label=k)
ax.set_xticks(range(0, cl.T, 3)); ax.set_xticklabels([str(m) for m in panel.months[::3]], rotation=45)
ax.set_ylabel("이용 기업 비율 (%)"); ax.set_title("월별 이용 비율 (코드 > 0), P36"); ax.legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.show()
rows = [{"변수": k, **{f"{y}년(%)": round(float(v[12 * i: 12 * i + 12].mean()), 1) for i, y in enumerate((2023, 2024, 2025))}} for k, v in use.items()]
cl.show_table(pd.DataFrame(rows), "연도별 평균 이용 비율 (P36)", OUT / "nb03", "usage_by_year")

## 5. 주 온라인 채널의 이동: 2023년 → 2025년

In [ ]:
def dominant(mask, yr):
    sl = slice(12 * yr, 12 * yr + 12)
    A = np.vstack([np.nanmean(cl.approx_count(cl.codes_float(panel.cnt[c][mask][:, sl], panel.observed[mask][:, sl]), cl.DEFAULT_CAP), axis=1)
                   for c in cl.ONLINE]).T
    A = np.nan_to_num(A, nan=-1.0)
    best = A.argmax(axis=1)
    return np.where(A.max(axis=1) <= 0, 3, best)

labs = ["인터넷", "스마트", "폰", "없음"]
res = {}
for pname, m in pops.items():
    d23, d25 = dominant(m, 0), dominant(m, 2)
    ct = pd.crosstab(pd.Series(np.array(labs)[d23], name="2023 주 채널"), pd.Series(np.array(labs)[d25], name="2025 주 채널")).reindex(index=labs, columns=labs, fill_value=0)
    res[pname] = ct
    show = ct.copy().astype(object)
    for i in show.index:
        for j in show.columns:
            show.loc[i, j] = cl.sup(ct.loc[i, j])
    cl.show_table(show.reset_index(), f"주 온라인 채널 전환표 ({pname})", OUT / "nb03", f"transition_{pname.replace('+', 'p')}")
ct = res["P36"]
sw = ct.loc["인터넷", "스마트"] / max(ct.loc["인터넷"].sum(), 1) * 100
print(f"P36: 2023년 인터넷 주 이용 기업 중 2025년 스마트로 바뀐 비율 {sw:.1f}%")

## 6. 채널 이동 이벤트의 동시 발생 (lift) — 탐색적 분석
분기 말 8개 창(2024.03 … 2025.12)에서 각 채널의 **최근 3개월 평균 코드 − 1년 전 같은 달 평균 코드** 를 구합니다. **두 해 모두 관측된 달만 짝지어** 평균하므로(쌍 맞춤) 빈 달이 있어도 '같은 달끼리' 비교하고, 짝지은 달이 2개 미만인 창은 판정하지 않습니다.
변화가 ≤ −1 이면 *감소*, ≥ +1 이면 *증가* 이벤트입니다 (임계 ±1 은 계획서에서 사전 확정).

| 지표 | 정의 | 읽는 법 |
|---|---|---|
| **반대 방향 lift** | P(A 감소 & B 증가) ÷ (P(A 감소)·P(B 증가)) | 1보다 크면 독립일 때보다 *이동*이 더 자주 동시에 일어남. **확증 판정은 이 지표(전체 창)로만 합니다** |
| 총 활동 유지 창만의 반대 방향 lift (**보조**) | 국내 5채널 합(I1)이 ±20% 이내로 유지된 창만 대상 | 총합을 고정하면 구성 채널 사이에 음의 연동이 **인위적으로 생겨 실제보다 커 보일 수 있습니다.** 판정에는 쓰지 않고 전체 창 결과와 **나란히** 읽습니다 |
| 같은 방향 lift | P(A 감소 & B 감소) ÷ (P(A 감소)·P(B 감소)) | 함께 줄어드는 정도 (전반적 위축·크기 효과) |
| 다음 창 lift | A 감소가 있은 *다음 창*에서 B 증가 | 같은 창 동시 발생과 구별되는 선후 관계 |

- **확증 vs 탐색**: **인터넷 감소 → 스마트 증가 한 쌍만 확증 대상**(사전 지정, 95% 신뢰구간). **확증 지지 = 전체 창 반대 방향 lift의 95% CI 하한 > 1 이고, 점추정이 같은 방향 lift보다 큼.**
  나머지 19개 쌍은 **탐색으로만 보고**하며 해석·처방에는 쓰지 않습니다 (CI는 20쌍 Bonferroni 99.75%).
- 신뢰구간은 **기업 단위 재표집**(기업의 모든 창을 함께 뽑음)으로 구합니다. 99.75% CI의 극단 분위(0.125%)를 안정적으로 추정하려고 기본 **10,000회**를 씁니다 (몇 분 걸릴 수 있음).
- 이 지표는 **'이동 후보'** 를 보여 줄 뿐 **대체를 입증하지 않습니다.** 계절·공통 충격·규모 변화가 남아 있을 수 있습니다.

In [ ]:
ends = [14, 17, 20, 23, 26, 29, 32, 35]
THR, STABLE, K3 = 1.0, 0.20, 3
PRIMARY = ("인터넷뱅킹거래건수", "스마트뱅킹거래건수")      # 확증 대상: 인터넷 감소 -> 스마트 증가 (사전 지정)
BOOT = int(os.environ.get("CNT_BOOT", "10000"))        # 99.75% CI 의 극단 분위를 안정적으로 추정하려면 크게

def window_stats(mask):
    """창별 채널 코드 변화(쌍 맞춤)와 국내 5채널 합(I1, cap60)의 상대 변화."""
    Zs = {c: cl.codes_float(panel.cnt[c][mask], panel.observed[mask]) for c in cl.CH5}
    I1 = sum(cl.approx_count(Zs[c], cl.DEFAULT_CAP) for c in cl.CH5)
    need = cl.min_pairs(K3)
    D = {c: np.full((I1.shape[0], len(ends)), np.nan) for c in cl.CH5}
    R = np.full((I1.shape[0], len(ends)), np.nan)
    for j, e in enumerate(ends):
        for c in cl.CH5:
            a, b = cl.paired_means(Zs[c], e, K3, need)
            D[c][:, j] = a - b
        a, b = cl.paired_means(I1, e, K3, need)
        with np.errstate(divide="ignore", invalid="ignore"):
            R[:, j] = np.where(b > 0, (a - b) / b, np.nan)
    return D, R

def lf4(t, s):
    v, a, b, j = t[s], t[s + 1], t[s + 2], t[s + 3]
    return j * v / (a * b) if a > 0 and b > 0 else np.nan

def lf_same(t):
    return t[9] * t[0] / (t[1] * t[8]) if t[1] > 0 and t[8] > 0 else np.nan

def all4(t):
    return [lf4(t, 0), lf4(t, 4), lf_same(t), lf4(t, 10)]      # 반대, 총활동 유지 창 반대, 같은 방향, 다음 창

def lift_parts(D, A, Bc, stable):
    """기업별 개수 열: [V, nA, nBu, Jo | Vs, nAs, nBus, Jos | nBd, Js | Vl, nAl, nBul, Jl]"""
    valid = ~np.isnan(D[A]) & ~np.isnan(D[Bc])
    dA, dB, uB = (D[A] <= -THR) & valid, (D[Bc] <= -THR) & valid, (D[Bc] >= THR) & valid
    vs = valid & stable
    vl = valid[:, :-1] & valid[:, 1:]
    dAl, uBl = dA[:, :-1] & vl, uB[:, 1:] & vl
    return np.column_stack([
        valid.sum(1), dA.sum(1), uB.sum(1), (dA & uB).sum(1),
        vs.sum(1), (dA & vs).sum(1), (uB & vs).sum(1), (dA & uB & vs).sum(1),
        dB.sum(1), (dA & dB).sum(1),
        vl.sum(1), dAl.sum(1), uBl.sum(1), (dAl & uBl).sum(1)]).astype(float)

def lifts(mask, B=BOOT, seed=7):
    D, R = window_stats(mask)
    n = R.shape[0]
    stable = np.abs(R) <= STABLE
    rng = np.random.default_rng(seed)
    pairs = [(A, Bc) for A in cl.CH5 for Bc in cl.CH5 if A != Bc]
    Sm = {p: lift_parts(D, p[0], p[1], stable) for p in pairs}
    boots = {p: np.full((B, 4), np.nan) for p in pairs}
    for b in range(B):                                     # 기업 단위 재표집: 기업의 모든 창을 함께 뽑는다
        idx = rng.integers(0, n, n)
        for p in pairs:
            boots[p][b] = all4(Sm[p][idx].sum(0))
    rows = []
    for p in pairs:
        A, Bc = p
        primary = p == PRIMARY
        lo, hi = (2.5, 97.5) if primary else (0.125, 99.875)   # 탐색 쌍은 20쌍 Bonferroni (99.75%)
        tot = Sm[p].sum(0)
        pts = all4(tot)
        ci = [np.nanpercentile(boots[p][:, j], [lo, hi]) if np.isfinite(boots[p][:, j]).any() else np.array([np.nan, np.nan]) for j in range(4)]
        f = lambda j: f"{ci[j][0]:.2f}~{ci[j][1]:.2f}"
        rows.append({"A 감소": cl.SHORT[A], "B 증가/감소": cl.SHORT[Bc], "구분": "확증" if primary else "탐색(보고만)",
                     "유효 창": cl.sup(int(tot[0])), "동시 이벤트(반대)": cl.sup(int(tot[3])),
                     "반대 lift": round(pts[0], 2), "반대 CI": f(0),
                     "총활동 유지 창 반대 lift(보조)": round(pts[1], 2), "유지 CI": f(1),
                     "같은 방향 lift": round(pts[2], 2), "같은 CI": f(2),
                     "다음 창 lift": round(pts[3], 2), "다음 창 CI": f(3)})
    return pd.DataFrame(rows)

print(f"부트스트랩 {BOOT:,}회 (환경변수 CNT_BOOT 로 조정)")
L36 = lifts(pops["P36"]); L35 = lifts(pops["P35+"])
cl.show_table(L36, "채널 이동 lift (P36) — 확증 1쌍은 95% CI, 탐색 19쌍은 99.75% CI", OUT / "nb03", "lift_P36")
cl.show_table(L35, "채널 이동 lift (P35+)", OUT / "nb03", "lift_P35p")
def lo_ci(s): return float(s.split("~")[0])
conf = L36[L36["구분"] == "확증"].iloc[0]
CONF_SUPPORTED = bool(lo_ci(conf["반대 CI"]) > 1 and conf["반대 lift"] > conf["같은 방향 lift"])
print(f"[확증] 인터넷↓ → 스마트↑ (P36): 반대 lift {conf['반대 lift']} ({conf['반대 CI']}) vs 같은 방향 {conf['같은 방향 lift']} -> {'지지' if CONF_SUPPORTED else '미지지'}")
print(f"[보조] 총 활동 유지 창 반대 lift {conf['총활동 유지 창 반대 lift(보조)']} ({conf['유지 CI']}) — 총합 조건화로 인위적으로 커질 수 있어 판정에 쓰지 않음")
print("[탐색] 반대 lift 신뢰구간(99.75%) 하한이 1을 넘는 쌍 (P36, 보고만):")
expl = L36[(L36["구분"] != "확증") & (L36["반대 CI"].map(lo_ci) > 1)][["A 감소", "B 증가/감소", "반대 lift", "반대 CI", "같은 방향 lift"]]
print(expl.to_string(index=False) if len(expl) else "  없음")

## 7. 외환: 다른 채널과 독립된 축인가

In [ ]:
P = pops["P36"]
fx_used = np.zeros(P.sum(), dtype=bool)
rows = []
for c in cl.FX:
    u = ((panel.cnt[c][P] > 0) & panel.observed[P]).any(axis=1)
    fx_used |= u
    rows.append({"구분": cl.SHORT[c], "이용 경험 기업(%)": round(u.mean() * 100, 1)})
rows.append({"구분": "외환 수출 또는 수입", "이용 경험 기업(%)": round(fx_used.mean() * 100, 1)})
cl.show_table(pd.DataFrame(rows), "외환 이용 기업 비율 (36개월 중 한 번이라도, P36)", OUT / "nb03", "fx_usage")
sec = panel.attr["업종_대분류"][P]
lv = panel.attr_levels["업종_대분류"]
srow = []
for k, name in enumerate(lv):
    in_k = (sec == k).any(axis=1)
    if in_k.sum() >= 30:
        srow.append({"업종": name, "기업 수": cl.sup(int(in_k.sum())), "외환 이용 기업(%)": round(fx_used[in_k].mean() * 100, 1)})
srow = pd.DataFrame(srow).sort_values("외환 이용 기업(%)", ascending=False).head(8)
cl.show_table(srow, "업종별 외환 이용 비율 (기업 30개 이상인 업종 상위 8)", OUT / "nb03", "fx_by_sector")
fx_names = [cl.SHORT[c] for c in cl.FX]
dom_names = [cl.SHORT[c] for c in cl.CNT_COLS[:6]]
fx_corr = W36.loc[fx_names, dom_names].abs()
print(f"기업 내 상관: 외환 vs 국내 채널·자동이체 |상관| 최대 {float(np.nanmax(fx_corr.values)):.3f}")

## 8. 요약 저장

In [ ]:
def pick(L, a, b, col):
    r = L[(L["A 감소"] == a) & (L["B 증가/감소"] == b)]
    return float(r[col].iloc[0]) if len(r) else float("nan")

summary = {
    "within_corr_internet_smart_P36": float(W36.loc["인터넷", "스마트"]),
    "within_corr_internet_smart_P35p": float(W35.loc["인터넷", "스마트"]),
    "lift_opp_internet_down_smart_up_P36": pick(L36, "인터넷", "스마트", "반대 lift"),
    "lift_opp_stable_internet_down_smart_up_P36": pick(L36, "인터넷", "스마트", "총활동 유지 창 반대 lift(보조)"),
    "lift_same_internet_down_smart_down_P36": pick(L36, "인터넷", "스마트", "같은 방향 lift"),
    "confirmatory_ci_lower_P36": float(conf["반대 CI"].split("~")[0]),
    "confirmatory_supported_P36": CONF_SUPPORTED,
    "pca_var_explained_pct": [float(x) for x in ev["설명 비율(%)"].iloc[:3]],
    "fx_max_abs_within_corr_with_domestic": float(np.nanmax(fx_corr.values)),
    "pop_max_within_corr_diff": maxdiff,
    "fx_used_pct_P36": float(fx_used.mean() * 100),
    "synthetic": bool(panel.synthetic),
}
cl.save_json(WORK / "interim" / "nb03_summary.json", summary)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 체크포인트 (Claude에게 알려줄 것)
- [ ] 기업 내 상관에서 **음수**인 쌍이 있는가 (특히 인터넷–스마트, 창구–인터넷)
- [ ] 주성분 1·2의 설명 비율과, 적재값에서 온라인/오프라인/외환이 갈리는가
- [ ] **확증 1쌍(인터넷↓ → 스마트↑)**: 전체 창 반대 lift와 95% 신뢰구간, 같은 방향 lift와의 비교 (CI 하한 > 1 이고 반대 > 같은 방향이면 지지). 총 활동 유지 창 lift는 보조로만
- [ ] 탐색 쌍 중 반대 lift 신뢰구간(99.75%) 하한이 1을 넘는 쌍 (보고만, 해석·처방에는 쓰지 않음)
- [ ] P36 과 P35+ 의 차이 (기업 내 상관 최대 차이, lift 방향)
- [ ] 외환과 다른 채널의 기업 내 상관 최대값, 외환 이용 비율이 높은 업종

**다음**: `P3` 체크리스트 → `P4` 계획(활동 위축 탐지 규칙) 확인.